In [1]:
from pathlib import Path
import duckdb
import polars as pl

DATA_ROOT = Path(r"D:\business_entity_resolution/dataset")
WORK_DIR = DATA_ROOT / "work"
CANDIDATE_DIR = WORK_DIR / "candidates"

NORM_DIR = WORK_DIR / "normalized"

NORM_TRAIN_S1 = NORM_DIR / "train_source1.parquet"
NORM_TRAIN_S2 = NORM_DIR / "train_source2.parquet"

COMBINED_V2 = CANDIDATE_DIR / "combined_name_translit_candidates.parquet"
LABELED_FUZZY = CANDIDATE_DIR / "fuzzy_scored_labeled.parquet"
MISSED_PAIRS = CANDIDATE_DIR / "missed_true_pairs.parquet"

DUCKDB_PATH = WORK_DIR / "entity_resolution.duckdb"

con = duckdb.connect(str(DUCKDB_PATH))

con.execute("SET memory_limit='2GB';")
con.execute(f"SET temp_directory='{WORK_DIR / 'duckdb_temp'}';")

con.execute(f"""
CREATE OR REPLACE VIEW train_s1 AS
SELECT * FROM read_parquet('{NORM_TRAIN_S1}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW train_s2 AS
SELECT * FROM read_parquet('{NORM_TRAIN_S2}');
""")

print("Recovery complete")
print("S1:", con.execute("SELECT COUNT(*) FROM train_s1").fetchone()[0])
print("S2:", con.execute("SELECT COUNT(*) FROM train_s2").fetchone()[0])

Recovery complete
S1: 2206821
S2: 5034616


In [2]:
GROUND_TRUTH = Path(r"D:\business_entity_resolution\dataset\train\train_ground_truth.tsv")

gt = pl.read_csv(
    GROUND_TRUTH,
    separator="\t"
)

gt_pairs = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .fill_null("")
        .str.split(",")
        .alias("matched_ids")
    )
    .explode("matched_ids")
    .rename({"matched_ids": "candidate_entity_id"})
    .filter(
        pl.col("candidate_entity_id").str.len_chars() > 0
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id"
    ])
)

gt_s2 = gt_pairs.filter(
    pl.col("candidate_entity_id").str.starts_with("S2-")
)

con.register("gt_s2", gt_s2)

print("GT S2 pairs:", gt_s2.height)

C:\Users\KRISH\tmp\ipykernel_16832\3039906301.py:16: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("matched_ids")


GT S2 pairs: 3693619


In [3]:
LEGAL_TOKEN_SQL = """
'private','pvt','limited','ltd','llc','llp','inc','incorporated',
'corp','corporation','company','co','lp','pllc',
'प्राइवेट','लिमिटेड','प्रा','लि','एलएलपी',
'ప్రైవేట్','లిమిటెడ్','ఎల్ఎల్పి',
'ಪ್ರೈವೇಟ್','ಲಿಮಿಟೆಡ್','ಎಲ್ಎಲ್ಪಿ',
'பிரைவேட்','லிமிடெட்','எல்எல்பி',
'প্রাইভেট','লিমিটেড','এলএলপি',
'પ્રાઇવેટ','લિમિટેડ','એલએલપી'
"""

con.execute(f"""
CREATE OR REPLACE VIEW s1_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s1
WHERE name_norm <> ''
""")

con.execute(f"""
CREATE OR REPLACE VIEW s2_informative_anchor AS
SELECT
    entity_id,
    name_norm,
    country_norm,
    array_to_string(
        list_sort(
            list_filter(
                string_split(name_norm, ' '),
                x ->
                    x <> ''
                    AND x NOT IN ({LEGAL_TOKEN_SQL})
                    AND LENGTH(x) >= 3
            )
        ),
        ' '
    ) AS informative_name
FROM train_s2
WHERE name_norm <> ''
""")

print("Informative views ready")

Informative views ready


In [4]:
con.execute("""
CREATE OR REPLACE VIEW s2_informative_tokens AS
SELECT
    entity_id,
    country_norm,
    UNNEST(string_split(informative_name, ' ')) AS token
FROM s2_informative_anchor
WHERE informative_name <> ''
""")

con.execute("""
CREATE OR REPLACE VIEW s2_rare_tokens AS
SELECT
    token,
    country_norm,
    COUNT(*) AS token_freq
FROM s2_informative_tokens
GROUP BY token, country_norm
HAVING COUNT(*) <= 50
""")

print(
    con.execute(
        "SELECT COUNT(*) FROM s2_rare_tokens"
    ).fetchone()[0],
    "rare token-country combinations"
)

820684 rare token-country combinations


In [5]:
S2_RARE_TOKEN_PARQUET = CANDIDATE_DIR / "s2_rare_token_postings.parquet"

con.execute(f"""
COPY (
    SELECT
        t.entity_id,
        t.country_norm,
        t.token
    FROM s2_informative_tokens t
    JOIN s2_rare_tokens r
        ON t.token = r.token
       AND t.country_norm = r.country_norm
)
TO '{S2_RARE_TOKEN_PARQUET}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("Created:", S2_RARE_TOKEN_PARQUET)

Created: D:\business_entity_resolution\dataset\work\candidates\s2_rare_token_postings.parquet


In [6]:
con.execute(f"""
SELECT
    COUNT(*) AS rare_postings,
    COUNT(DISTINCT entity_id) AS s2_entities
FROM read_parquet('{S2_RARE_TOKEN_PARQUET}')
""").pl()

rare_postings,s2_entities
i64,i64
2021163,1793292


In [7]:
BATCH_SIZE = 50_000

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s1_batch AS
SELECT
    entity_id,
    country_norm,
    informative_name
FROM s1_informative_anchor
WHERE informative_name <> ''
LIMIT {BATCH_SIZE}
""")

print(
    "S1 batch:",
    con.execute("SELECT COUNT(*) FROM s1_batch").fetchone()[0]
)

S1 batch: 50000


In [8]:
con.execute("""
CREATE OR REPLACE VIEW s1_informative_tokens AS
SELECT
    s.entity_id AS source1_entity_id,
    s.country_norm,
    t.token
FROM s1_informative_anchor s
CROSS JOIN UNNEST(string_split(s.informative_name, ' ')) AS t(token)
WHERE s.informative_name <> ''
  AND s.informative_name IS NOT NULL
  AND t.token <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS token_rows,
            COUNT(DISTINCT source1_entity_id) AS s1_entities
        FROM s1_informative_tokens
    """).fetchdf()
)

   token_rows  s1_entities
0     5392776      2205573


In [9]:
# Benchmark direct token-to-token rare-token blocking on 50K S1 entities

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_rare_test AS
SELECT DISTINCT source1_entity_id
FROM s1_informative_tokens
LIMIT 50000
""")

test_candidates = con.execute(f"""
SELECT DISTINCT
    s1.source1_entity_id,
    s2.entity_id AS candidate_entity_id
FROM s1_informative_tokens s1
JOIN s1_rare_test b
    ON s1.source1_entity_id = b.source1_entity_id
JOIN read_parquet('{S2_RARE_TOKEN_PARQUET}') s2
    ON s1.country_norm = s2.country_norm
   AND s1.token = s2.token
""").fetchdf()

print("candidate pairs:", len(test_candidates))
print("S1 entities:", test_candidates["source1_entity_id"].nunique())

candidate pairs: 245219
S1 entities: 10715


In [10]:
# Measure rare-token blocker recall on the 50K benchmark

result = con.execute("""
WITH candidates AS (
    SELECT DISTINCT
        s1.source1_entity_id,
        s2.entity_id AS candidate_entity_id
    FROM s1_informative_tokens s1
    JOIN s1_rare_test b
        ON s1.source1_entity_id = b.source1_entity_id
    JOIN read_parquet(?) s2
        ON s1.country_norm = s2.country_norm
       AND s1.token = s2.token
)
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / NULLIF(COUNT(*), 0),
        4
    ) AS recall_pct
FROM gt_s2 g
JOIN s1_rare_test b
    ON g.source1_entity_id = b.source1_entity_id
LEFT JOIN candidates c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""", [str(S2_RARE_TOKEN_PARQUET)]).fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0     83851            15578     18.5782


In [11]:
# Inspect missed true pairs using first 3 alphabetic/name characters

print(
    con.execute("""
        SELECT
            COUNT(*) AS missed_pairs,
            COUNT(*) FILTER (
                WHERE LEFT(
                    regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
                )
                =
                LEFT(
                    regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 3
                )
            ) AS same_prefix3
        FROM read_parquet(?) m
        JOIN train_s1 s1
            ON m.source1_entity_id = s1.entity_id
        JOIN train_s2 s2
            ON m.candidate_entity_id = s2.entity_id
    """, [str(WORK_DIR / "candidates" / "missed_true_pairs.parquet")])
    .fetchdf()
)

   missed_pairs  same_prefix3
0       1877384       1182594


In [12]:
# Build S2 prefix-3 postings, capped at 100 candidates per country/prefix.
# We use the alphanumeric normalized name representation.

con.execute("""
CREATE OR REPLACE VIEW s2_name_prefix3 AS
SELECT
    entity_id,
    country_norm,
    LEFT(regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3) AS prefix3
FROM train_s2
WHERE name_norm <> ''
""")

con.execute("""
CREATE OR REPLACE VIEW s2_prefix3_postings AS
SELECT
    entity_id,
    country_norm,
    prefix3,
    ROW_NUMBER() OVER (
        PARTITION BY country_norm, prefix3
        ORDER BY entity_id
    ) AS rn
FROM s2_name_prefix3
WHERE prefix3 <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT prefix3) AS prefixes,
            MAX(rn) AS max_posting
        FROM s2_prefix3_postings
    """).fetchdf()
)

   postings  prefixes  max_posting
0   4578856     27173        46901


In [13]:
# Measure how much prefix-3 + name-length bucket reduces the large postings.

con.execute("""
CREATE OR REPLACE VIEW s2_prefix3_len AS
SELECT
    entity_id,
    country_norm,
    LEFT(regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3) AS prefix3,
    LEAST(LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) / 5, 10)::INTEGER AS len_bucket
FROM train_s2
WHERE name_norm <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT prefix3 || '|' || country_norm || '|' || len_bucket) AS blocks,
            MAX(cnt) AS max_block,
            ROUND(AVG(cnt), 2) AS avg_block,
            APPROX_QUANTILE(cnt, 0.95) AS p95_block
        FROM (
            SELECT
                prefix3,
                country_norm,
                len_bucket,
                COUNT(*) AS cnt
            FROM s2_prefix3_len
            WHERE prefix3 <> ''
            GROUP BY prefix3, country_norm, len_bucket
        )
    """).fetchdf()
)

   postings  blocks  max_block  avg_block  p95_block
0    164963  164963      14901      27.76         70


In [14]:
# Test prefix-3 + country + length bucket recall on missed true pairs

result = con.execute("""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        LEFT(
            regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
        ) AS prefix3,
        LEAST(
            LENGTH(regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER AS len_bucket,
        s1.country_norm
    FROM read_parquet(?) m
    JOIN train_s1 s1
      ON m.source1_entity_id = s1.entity_id
),
s2_keys AS (
    SELECT
        entity_id,
        country_norm,
        LEFT(
            regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3
        ) AS prefix3,
        LEAST(
            LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER AS len_bucket
    FROM train_s2
)
SELECT
    COUNT(*) AS missed_pairs,
    COUNT(*) FILTER (
        WHERE s2.entity_id IS NOT NULL
    ) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE s2.entity_id IS NOT NULL
        ) / COUNT(*),
        4
    ) AS recall_pct
FROM missed m
LEFT JOIN s2_keys s2
  ON m.candidate_entity_id = s2.entity_id
 AND m.country_norm = s2.country_norm
 AND m.prefix3 = s2.prefix3
 AND m.len_bucket = s2.len_bucket
""", [str(WORK_DIR / "candidates" / "missed_true_pairs.parquet")]).fetchdf()

print(result)

   missed_pairs  recovered_pairs  recall_pct
0       1877384           617492     32.8911


In [15]:
# Measure how many missed true pairs this blocker recovers,
# and how many are genuinely new relative to COMBINED_V2.

result = con.execute("""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        LEFT(
            regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
        ) AS prefix3,
        LEAST(
            LENGTH(regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER AS len_bucket,
        s1.country_norm
    FROM read_parquet(?) m
    JOIN train_s1 s1
      ON m.source1_entity_id = s1.entity_id
),
s2_keys AS (
    SELECT
        entity_id,
        country_norm,
        LEFT(
            regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3
        ) AS prefix3,
        LEAST(
            LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER AS len_bucket
    FROM train_s2
),
prefix_recovered AS (
    SELECT DISTINCT
        m.source1_entity_id,
        m.candidate_entity_id
    FROM missed m
    JOIN s2_keys s2
      ON m.candidate_entity_id = s2.entity_id
     AND m.country_norm = s2.country_norm
     AND m.prefix3 = s2.prefix3
     AND m.len_bucket = s2.len_bucket
),
combined AS (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet(?)
)
SELECT
    COUNT(*) AS prefix_recovered,
    COUNT(*) FILTER (
        WHERE c.candidate_entity_id IS NULL
    ) AS genuinely_new,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE c.candidate_entity_id IS NULL
        ) / COUNT(*),
        4
    ) AS new_pct
FROM prefix_recovered p
LEFT JOIN combined c
  ON p.source1_entity_id = c.source1_entity_id
 AND p.candidate_entity_id = c.candidate_entity_id
""", [
    str(WORK_DIR / "candidates" / "missed_true_pairs.parquet"),
    str(COMBINED_V2)
]).fetchdf()

print(result)

   prefix_recovered  genuinely_new  new_pct
0            617492         617492    100.0


In [16]:
# Combined recall: COMBINED_V2 + prefix3+length blocker

result = con.execute("""
WITH prefix_recovered AS (
    SELECT DISTINCT
        m.source1_entity_id,
        m.candidate_entity_id
    FROM read_parquet(?) m
    JOIN train_s1 s1
      ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
      ON m.candidate_entity_id = s2.entity_id
    WHERE
        s1.country_norm = s2.country_norm
        AND LEFT(
            regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
        )
        =
        LEFT(
            regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 3
        )
        AND LEAST(
            LENGTH(regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER
        =
        LEAST(
            LENGTH(regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g')) / 5,
            10
        )::INTEGER
),
combined AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_parquet(?)

    UNION

    SELECT source1_entity_id, candidate_entity_id
    FROM prefix_recovered
)
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        4
    ) AS recall_pct
FROM gt_s2 g
LEFT JOIN combined c
  ON g.source1_entity_id = c.source1_entity_id
 AND g.candidate_entity_id = c.candidate_entity_id
""", [
    str(WORK_DIR / "candidates" / "missed_true_pairs.parquet"),
    str(COMBINED_V2)
]).fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0   3693619          2433727       65.89


In [17]:
# Safer DuckDB settings for large candidate generation

con.execute("SET memory_limit='2GB';")
con.execute("SET threads=2;")
con.execute("SET preserve_insertion_order=false;")

PREFIX_DIR = CANDIDATE_DIR / "prefix3_length_parts"
PREFIX_DIR.mkdir(parents=True, exist_ok=True)

print("Ready:", PREFIX_DIR)

Ready: D:\business_entity_resolution\dataset\work\candidates\prefix3_length_parts


In [18]:
# PREFIX_CANDIDATES = CANDIDATE_DIR / "prefix3_length_candidates.parquet"

# con.execute(f"""
# COPY (
#     SELECT DISTINCT
#         s1.entity_id AS source1_entity_id,
#         s2.entity_id AS candidate_entity_id
#     FROM train_s1 s1
#     JOIN train_s2 s2
#       ON s1.country_norm = s2.country_norm
#      AND LEFT(
#             regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
#          )
#          =
#          LEFT(
#             regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 3
#          )
#      AND LEAST(
#             LENGTH(regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g')) / 5,
#             10
#          )::INTEGER
#          =
#          LEAST(
#             LENGTH(regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g')) / 5,
#             10
#          )::INTEGER
#     WHERE s1.name_norm <> ''
#       AND s2.name_norm <> ''
# ) TO '{PREFIX_CANDIDATES}'
# (FORMAT PARQUET, COMPRESSION ZSTD)
# """)

# print(PREFIX_CANDIDATES)

In [19]:
# Clean DuckDB restart after the OOM

con.close()

con = duckdb.connect(str(DUCKDB_PATH))

con.execute("SET memory_limit='2GB';")
con.execute("SET threads=2;")
con.execute("SET preserve_insertion_order=false;")
con.execute(f"SET temp_directory='{WORK_DIR / 'duckdb_temp'}';")

con.execute(f"""
CREATE OR REPLACE VIEW train_s1 AS
SELECT * FROM read_parquet('{NORM_TRAIN_S1}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW train_s2 AS
SELECT * FROM read_parquet('{NORM_TRAIN_S2}')
""")

con.register("gt_s2", gt_s2)

print(
    con.execute("""
        SELECT
            (SELECT COUNT(*) FROM train_s1) AS s1_rows,
            (SELECT COUNT(*) FROM train_s2) AS s2_rows,
            (SELECT COUNT(*) FROM gt_s2) AS gt_pairs
    """).fetchdf()
)

   s1_rows  s2_rows  gt_pairs
0  2206821  5034616   3693619


In [20]:
# Prepare batched prefix candidate generation

PREFIX_DIR = CANDIDATE_DIR / "prefix3_length_parts"
PREFIX_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 25_000

# Build S1 keys once
con.execute("""
CREATE OR REPLACE VIEW s1_prefix3_length AS
SELECT
    entity_id AS source1_entity_id,
    country_norm,
    LEFT(
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3
    ) AS prefix3,
    LEAST(
        LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) / 5,
        10
    )::INTEGER AS len_bucket
FROM train_s1
WHERE name_norm <> ''
""")

# Build S2 keys once
con.execute("""
CREATE OR REPLACE VIEW s2_prefix3_length AS
SELECT
    entity_id AS candidate_entity_id,
    country_norm,
    LEFT(
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 3
    ) AS prefix3,
    LEAST(
        LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) / 5,
        10
    )::INTEGER AS len_bucket
FROM train_s2
WHERE name_norm <> ''
""")

s1_count = con.execute("""
    SELECT COUNT(*) FROM s1_prefix3_length
""").fetchone()[0]

print("S1 entities with names:", s1_count)
print("Batch size:", BATCH_SIZE)
print("Number of batches:", (s1_count + BATCH_SIZE - 1) // BATCH_SIZE)

S1 entities with names: 2206821
Batch size: 25000
Number of batches: 89


In [21]:
# Generate and persist prefix candidates for batch 0

batch_id = 0
offset = batch_id * BATCH_SIZE
output_path = PREFIX_DIR / f"part_{batch_id:03d}.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        s1.source1_entity_id,
        s2.candidate_entity_id
    FROM (
        SELECT *
        FROM s1_prefix3_length
        ORDER BY source1_entity_id
        LIMIT {BATCH_SIZE}
        OFFSET {offset}
    ) s1
    JOIN s2_prefix3_length s2
      ON s1.country_norm = s2.country_norm
     AND s1.prefix3 = s2.prefix3
     AND s1.len_bucket = s2.len_bucket
) TO '{output_path}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Written:", output_path)
print(
    con.execute(f"""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates
        FROM read_parquet('{output_path}')
    """).fetchdf()
)

Written: D:\business_entity_resolution\dataset\work\candidates\prefix3_length_parts\part_000.parquet
   candidate_pairs  s1_with_candidates
0         31553687               24886


In [22]:
# Cap each prefix/length blocking bucket at 100 S2 records

con.execute("""
CREATE OR REPLACE VIEW s2_prefix3_length_capped AS
SELECT
    candidate_entity_id,
    country_norm,
    prefix3,
    len_bucket
FROM (
    SELECT
        candidate_entity_id,
        country_norm,
        prefix3,
        len_bucket,
        ROW_NUMBER() OVER (
            PARTITION BY country_norm, prefix3, len_bucket
            ORDER BY candidate_entity_id
        ) AS rn
    FROM s2_prefix3_length
)
WHERE rn <= 100
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT country_norm || '|' || prefix3 || '|' || CAST(len_bucket AS VARCHAR))
                AS blocks,
            MAX(cnt) AS max_block
        FROM (
            SELECT
                country_norm,
                prefix3,
                len_bucket,
                COUNT(*) AS cnt
            FROM s2_prefix3_length_capped
            GROUP BY country_norm, prefix3, len_bucket
        )
    """).fetchdf()
)

   postings  blocks  max_block
0    164965  164965        100


In [23]:
# Generate capped prefix candidates for batch 0

batch_id = 0
offset = batch_id * BATCH_SIZE
output_path = PREFIX_DIR / f"capped_part_{batch_id:03d}.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        s1.source1_entity_id,
        s2.candidate_entity_id
    FROM (
        SELECT *
        FROM s1_prefix3_length
        ORDER BY source1_entity_id
        LIMIT {BATCH_SIZE}
        OFFSET {offset}
    ) s1
    JOIN s2_prefix3_length_capped s2
      ON s1.country_norm = s2.country_norm
     AND s1.prefix3 = s2.prefix3
     AND s1.len_bucket = s2.len_bucket
) TO '{output_path}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Written:", output_path)

print(
    con.execute(f"""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM read_parquet('{output_path}')
    """).fetchdf()
)

Written: D:\business_entity_resolution\dataset\work\candidates\prefix3_length_parts\capped_part_000.parquet
   candidate_pairs  s1_with_candidates  avg_candidates_per_s1
0          2158235               24886                  86.72


In [24]:
# Recall of capped prefix blocker on batch 0

result = con.execute(f"""
WITH candidates AS (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet('{PREFIX_DIR / "capped_part_000.parquet"}')
)
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        4
    ) AS recall_pct
FROM gt_s2 g
JOIN (
    SELECT DISTINCT source1_entity_id
    FROM s1_prefix3_length
    ORDER BY source1_entity_id
    LIMIT {BATCH_SIZE}
) b
    ON g.source1_entity_id = b.source1_entity_id
LEFT JOIN candidates c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0     41605             6949     16.7023


In [25]:
# Build stronger name signature

con.execute("""
CREATE OR REPLACE VIEW s2_name_signature AS
SELECT
    entity_id AS candidate_entity_id,
    country_norm,
    LEFT(x, 3) || RIGHT(x, 2) AS name_sig,
    LEAST(LENGTH(x) / 5, 10)::INTEGER AS len_bucket
FROM (
    SELECT
        entity_id,
        country_norm,
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g') AS x
    FROM train_s2
    WHERE name_norm <> ''
)
WHERE LENGTH(x) >= 5
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT country_norm || '|' || name_sig || '|' ||
                  CAST(len_bucket AS VARCHAR)) AS blocks,
            MAX(cnt) AS max_block,
            ROUND(AVG(cnt), 2) AS avg_block,
            APPROX_QUANTILE(cnt, 0.95) AS p95_block
        FROM (
            SELECT
                country_norm,
                name_sig,
                len_bucket,
                COUNT(*) AS cnt
            FROM s2_name_signature
            GROUP BY country_norm, name_sig, len_bucket
        )
    """).fetchdf()
)

   postings  blocks  max_block  avg_block  p95_block
0    962753  962753       9626       4.74         14


In [26]:
# Build the same stronger signature for S1

con.execute("""
CREATE OR REPLACE VIEW s1_name_signature AS
SELECT
    entity_id AS source1_entity_id,
    country_norm,
    LEFT(x, 3) || RIGHT(x, 2) AS name_sig,
    LEAST(LENGTH(x) / 5, 10)::INTEGER AS len_bucket
FROM (
    SELECT
        entity_id,
        country_norm,
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g') AS x
    FROM train_s1
    WHERE name_norm <> ''
)
WHERE LENGTH(x) >= 5
""")

# Test recall specifically on the true pairs missed by COMBINED_V2

result = con.execute("""
SELECT
    COUNT(*) AS missed_pairs,
    COUNT(*) FILTER (
        WHERE
            s1.country_norm = s2.country_norm
            AND LEFT(
                regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 3
            )
            ||
            RIGHT(
                regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 2
            )
            =
            LEFT(
                regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 3
            )
            ||
            RIGHT(
                regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 2
            )
            AND LEAST(
                LENGTH(regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g')) / 5,
                10
            )::INTEGER
            =
            LEAST(
                LENGTH(regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g')) / 5,
                10
            )::INTEGER
    ) AS recovered_pairs
FROM read_parquet(?) m
JOIN train_s1 s1
    ON m.source1_entity_id = s1.entity_id
JOIN train_s2 s2
    ON m.candidate_entity_id = s2.entity_id
""", [str(WORK_DIR / "candidates" / "missed_true_pairs.parquet")]).fetchdf()

result["recall_pct"] = (
    100 * result["recovered_pairs"] / result["missed_pairs"]
)

print(result)

   missed_pairs  recovered_pairs  recall_pct
0       1877384           346044   18.432244


In [27]:
# Test character 3-gram overlap on missed true pairs
# using DuckDB-compatible SUBSTR + range()

result = con.execute("""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g') AS s1_name,
        regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g') AS s2_name
    FROM read_parquet(?) m
    JOIN train_s1 s1
        ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
        ON m.candidate_entity_id = s2.entity_id
),
grams AS (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        s1_name,
        s2_name,
        (
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT
                    SUBSTR(s1_name, i, 3) AS gram
                FROM range(1, GREATEST(LENGTH(s1_name) - 2, 1)) r(i)
                WHERE LENGTH(s1_name) >= 3
            ) a
            JOIN (
                SELECT DISTINCT
                    SUBSTR(s2_name, j, 3) AS gram
                FROM range(1, GREATEST(LENGTH(s2_name) - 2, 1)) r(j)
                WHERE LENGTH(s2_name) >= 3
            ) b
            ON a.gram = b.gram
        ) AS shared_3grams
    FROM missed
)
SELECT
    COUNT(*) AS missed_pairs,
    COUNT(*) FILTER (
        WHERE shared_3grams > 0
    ) AS shared_3gram_pairs,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE shared_3grams > 0
        ) / COUNT(*),
        4
    ) AS recall_pct
FROM grams
""", [str(WORK_DIR / "candidates" / "missed_true_pairs.parquet")]).fetchdf()

print(result)

   missed_pairs  shared_3gram_pairs  recall_pct
0       1877384             1487075       79.21


In [28]:
# Build S2 character 3-gram postings.
# First inspect gram frequencies so we can choose a safe rarity cutoff.

con.execute("""
CREATE OR REPLACE VIEW s2_name_3grams AS
SELECT DISTINCT
    entity_id AS candidate_entity_id,
    country_norm,
    SUBSTR(
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g'),
        r.i,
        3
    ) AS gram
FROM train_s2
CROSS JOIN range(1, 100) r(i)
WHERE LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) >= r.i + 2
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS total_postings,
            COUNT(DISTINCT gram) AS unique_grams,
            APPROX_QUANTILE(cnt, 0.50) AS median_posting,
            APPROX_QUANTILE(cnt, 0.90) AS p90_posting,
            APPROX_QUANTILE(cnt, 0.95) AS p95_posting,
            MAX(cnt) AS max_posting
        FROM (
            SELECT
                gram,
                COUNT(*) AS cnt
            FROM s2_name_3grams
            GROUP BY gram
        )
    """).fetchdf()
)

   total_postings  unique_grams  median_posting  p90_posting  p95_posting  \
0           36804         36804              33         2332         7783   

   max_posting  
0       820873  


In [29]:
print(
    con.execute(f"""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT gram) AS grams,
            (
                SELECT COUNT(DISTINCT candidate_entity_id)
                FROM read_parquet('{S2_3GRAM_RARE}')
            ) AS s2_entities,
            MAX(cnt) AS max_posting
        FROM (
            SELECT
                gram,
                COUNT(*) AS cnt
            FROM read_parquet('{S2_3GRAM_RARE}')
            GROUP BY gram
        )
    """).fetchdf()
)

print("Saved:", S2_3GRAM_RARE)

NameError: name 'S2_3GRAM_RARE' is not defined

In [30]:
# Build S1 character 3-gram postings for a 50K benchmark

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_3gram_test AS
SELECT DISTINCT
    s.source1_entity_id,
    s.country_norm,
    SUBSTR(
        regexp_replace(s.name_norm, '[^a-z0-9]', '', 'g'),
        r.i,
        3
    ) AS gram
FROM (
    SELECT
        entity_id AS source1_entity_id,
        country_norm,
        name_norm
    FROM train_s1
    ORDER BY entity_id
    LIMIT 50000
) s
CROSS JOIN range(1, 200) r(i)
WHERE LENGTH(regexp_replace(s.name_norm, '[^a-z0-9]', '', 'g')) >= r.i + 2
  AND SUBSTR(
        regexp_replace(s.name_norm, '[^a-z0-9]', '', 'g'),
        r.i,
        3
      ) <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS token_rows,
            COUNT(DISTINCT source1_entity_id) AS s1_entities,
            COUNT(DISTINCT gram) AS grams
        FROM s1_3gram_test
    """).fetchdf()
)

   token_rows  s1_entities  grams
0      949635        50000  12401


In [31]:
# Benchmark rare 3-gram blocking on the 50K S1 sample

con.execute("""
CREATE OR REPLACE TEMP TABLE gram_candidates_test AS
SELECT DISTINCT
    s1.source1_entity_id,
    s2.candidate_entity_id
FROM s1_3gram_test s1
JOIN read_parquet(?) s2
    ON s1.country_norm = s2.country_norm
   AND s1.gram = s2.gram
""", [str(S2_3GRAM_RARE)])

print(
    con.execute("""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM gram_candidates_test
    """).fetchdf()
)

print(
    con.execute("""
        SELECT
            COUNT(*) AS gt_pairs,
            COUNT(c.candidate_entity_id) AS recovered_pairs,
            ROUND(
                100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
                4
            ) AS recall_pct
        FROM gt_s2 g
        JOIN (
            SELECT DISTINCT source1_entity_id
            FROM s1_3gram_test
        ) b
            ON g.source1_entity_id = b.source1_entity_id
        LEFT JOIN gram_candidates_test c
            ON g.source1_entity_id = c.source1_entity_id
           AND g.candidate_entity_id = c.candidate_entity_id
    """).fetchdf()
)

NameError: name 'S2_3GRAM_RARE' is not defined

In [32]:
# Prepare a 10K S1 fuzzy-name benchmark.
# We use a length-aware window to avoid unrestricted fuzzy comparison.

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_fuzzy_test AS
SELECT
    entity_id AS source1_entity_id,
    country_norm,
    name_norm,
    LENGTH(regexp_replace(name_norm, '[^a-z0-9]', '', 'g')) AS name_len
FROM train_s1
ORDER BY entity_id
LIMIT 10000
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS s1_entities,
            AVG(name_len) AS avg_name_len
        FROM s1_fuzzy_test
    """).fetchdf()
)

   s1_entities  avg_name_len
0        10000       21.0475


In [ ]:
# # Benchmark length-aware fuzzy name retrieval on 10K S1 entities

# con.execute("""
# CREATE OR REPLACE TEMP TABLE fuzzy_name_test AS
# SELECT
#     s1.source1_entity_id,
#     s2.entity_id AS candidate_entity_id,
#     jaro_winkler_similarity(s1.name_norm, s2.name_norm) AS name_jw
# FROM s1_fuzzy_test s1
# JOIN train_s2 s2
#     ON s1.country_norm = s2.country_norm
#    AND LENGTH(s2.name_norm)
#        BETWEEN CAST(s1.name_len * 0.70 AS INTEGER)
#            AND CAST(s1.name_len * 1.30 AS INTEGER)
#    AND s1.name_norm <> ''
#    AND s2.name_norm <> ''
# """)

# print(
#     con.execute("""
#         SELECT
#             COUNT(*) AS candidate_pairs,
#             COUNT(DISTINCT source1_entity_id) AS s1_entities,
#             ROUND(
#                 COUNT(*) * 1.0 /
#                 COUNT(DISTINCT source1_entity_id), 2
#             ) AS avg_candidates_per_s1
#         FROM fuzzy_name_test
#     """).fetchdf()
# )

# print(
#     con.execute("""
#         SELECT
#             COUNT(*) AS gt_pairs,
#             COUNT(c.candidate_entity_id) AS recovered_pairs,
#             ROUND(
#                 100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
#                 4
#             ) AS recall_pct
#         FROM gt_s2 g
#         JOIN s1_fuzzy_test b
#             ON g.source1_entity_id = b.source1_entity_id
#         LEFT JOIN fuzzy_name_test c
#             ON g.source1_entity_id = c.source1_entity_id
#            AND g.candidate_entity_id = c.candidate_entity_id
#     """).fetchdf()
# )

RuntimeError: Query interrupted

In [33]:
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

print("scikit-learn:", sklearn.__version__)
print("TF-IDF available:", TfidfVectorizer is not None)

scikit-learn: 1.9.1
TF-IDF available: True


In [34]:
# Count remaining missed true pairs by name similarity and address similarity

result = con.execute("""
SELECT
    CASE
        WHEN name_jw >= 0.95 THEN '0.95-1.00'
        WHEN name_jw >= 0.90 THEN '0.90-0.95'
        WHEN name_jw >= 0.80 THEN '0.80-0.90'
        WHEN name_jw >= 0.70 THEN '0.70-0.80'
        WHEN name_jw >= 0.50 THEN '0.50-0.70'
        ELSE '<0.50'
    END AS name_bucket,

    CASE
        WHEN address_jw >= 0.90 THEN '0.90-1.00'
        WHEN address_jw >= 0.75 THEN '0.75-0.90'
        WHEN address_jw >= 0.50 THEN '0.50-0.75'
        ELSE '<0.50'
    END AS address_bucket,

    COUNT(*) AS pairs

FROM read_parquet(
    'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
)

GROUP BY name_bucket, address_bucket
ORDER BY pairs DESC
""").fetchdf()

print(result)

   name_bucket address_bucket   pairs
0    0.90-0.95      0.90-1.00  240348
1    0.95-1.00      0.90-1.00  189910
2    0.80-0.90      0.90-1.00  177574
3    0.90-0.95      0.75-0.90  150613
4        <0.50      0.75-0.90  127208
5    0.80-0.90      0.75-0.90  122740
6    0.95-1.00      0.75-0.90  117722
7        <0.50      0.50-0.75  114855
8        <0.50      0.90-1.00  113834
9    0.90-0.95      0.50-0.75   84624
10   0.95-1.00      0.50-0.75   66742
11   0.80-0.90      0.50-0.75   64915
12   0.70-0.80      0.90-1.00   53187
13   0.50-0.70      0.90-1.00   44603
14   0.70-0.80      0.75-0.90   41803
15   0.50-0.70      0.75-0.90   34322
16   0.90-0.95          <0.50   26402
17   0.70-0.80      0.50-0.75   23489
18   0.95-1.00          <0.50   21364
19   0.50-0.70      0.50-0.75   20885
20   0.80-0.90          <0.50   17860
21       <0.50          <0.50   14577
22   0.70-0.80          <0.50    4735
23   0.50-0.70          <0.50    3072


In [35]:
# Address-token diagnostic on 100K missed pairs

result = con.execute("""
WITH sample AS (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        address_jw
    FROM read_parquet(
        'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
    )
    USING SAMPLE 100000
),
pairs AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        list_distinct(
            list_filter(
                string_split(
                    regexp_replace(s1.address_norm, '[^a-z0-9]', ' ', 'g'),
                    ' '
                ),
                x -> LENGTH(x) >= 3
            )
        ) AS s1_tokens,
        list_distinct(
            list_filter(
                string_split(
                    regexp_replace(s2.address_norm, '[^a-z0-9]', ' ', 'g'),
                    ' '
                ),
                x -> LENGTH(x) >= 3
            )
        ) AS s2_tokens
    FROM sample m
    JOIN train_s1 s1
        ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
        ON m.candidate_entity_id = s2.entity_id
)
SELECT
    COUNT(*) AS sample_pairs,
    COUNT(*) FILTER (
        WHERE list_has_any(s1_tokens, s2_tokens)
    ) AS shared_token_pairs,
    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE list_has_any(s1_tokens, s2_tokens)
        ) / COUNT(*),
        2
    ) AS shared_token_pct
FROM pairs
""").fetchdf()

print(result)

   sample_pairs  shared_token_pairs  shared_token_pct
0        100000               96301              96.3


In [36]:
# Inspect S2 address-token frequencies

con.execute("""
CREATE OR REPLACE VIEW s2_address_tokens AS
SELECT DISTINCT
    entity_id AS candidate_entity_id,
    country_norm,
    token
FROM (
    SELECT
        entity_id,
        country_norm,
        UNNEST(
            string_split(
                regexp_replace(address_norm, '[^a-z0-9]', ' ', 'g'),
                ' '
            )
        ) AS token
    FROM train_s2
)
WHERE LENGTH(token) >= 3
  AND token <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT token) AS unique_tokens,
            APPROX_QUANTILE(cnt, 0.50) AS median_posting,
            APPROX_QUANTILE(cnt, 0.90) AS p90_posting,
            APPROX_QUANTILE(cnt, 0.95) AS p95_posting,
            MAX(cnt) AS max_posting
        FROM (
            SELECT
                country_norm,
                token,
                COUNT(*) AS cnt
            FROM s2_address_tokens
            GROUP BY country_norm, token
        )
    """).fetchdf()
)

   postings  unique_tokens  median_posting  p90_posting  p95_posting  \
0    638902         606514               2           22           60   

   max_posting  
0       422877  


In [37]:
# Build rare address-token postings for S2
# Keep tokens appearing in at most 50 S2 records per country.

S2_RARE_ADDRESS = CANDIDATE_DIR / "s2_rare_address_token_postings.parquet"

con.execute(f"""
COPY (
    WITH token_freq AS (
        SELECT
            country_norm,
            token,
            COUNT(*) AS freq
        FROM s2_address_tokens
        GROUP BY country_norm, token
        HAVING COUNT(*) <= 50
    )
    SELECT
        a.candidate_entity_id,
        a.country_norm,
        a.token
    FROM s2_address_tokens a
    JOIN token_freq f
      ON a.country_norm = f.country_norm
     AND a.token = f.token
) TO '{S2_RARE_ADDRESS}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")



BinderException: Binder Error: Referenced column "candidate_entity_id" not found in FROM clause!
Candidate bindings: "cnt"

LINE 5:             COUNT(DISTINCT candidate_entity_id) AS s2_entities,
                                   ^

In [38]:
print(
    con.execute(f"""
        SELECT
            COUNT(*) AS postings,
            COUNT(DISTINCT token) AS rare_tokens,
            (
                SELECT COUNT(DISTINCT candidate_entity_id)
                FROM read_parquet('{S2_RARE_ADDRESS}')
            ) AS s2_entities,
            MAX(cnt) AS max_posting
        FROM (
            SELECT
                token,
                COUNT(*) AS cnt
            FROM read_parquet('{S2_RARE_ADDRESS}')
            GROUP BY token
        )
    """).fetchdf()
)

print("Saved:", S2_RARE_ADDRESS)

   postings  rare_tokens  s2_entities  max_posting
0    583826       583826      2191884           96
Saved: D:\business_entity_resolution\dataset\work\candidates\s2_rare_address_token_postings.parquet


In [39]:
# Test rare-address-token coverage on 100K missed true pairs

result = con.execute("""
WITH sample AS (
    SELECT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet(
        'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
    )
    USING SAMPLE 100000
),
s1_tokens AS (
    SELECT DISTINCT
        s.source1_entity_id,
        t.token
    FROM sample m
    JOIN train_s1 s
        ON m.source1_entity_id = s.entity_id
    CROSS JOIN UNNEST(
        string_split(
            regexp_replace(s.address_norm, '[^a-z0-9]', ' ', 'g'),
            ' '
        )
    ) AS t(token)
    WHERE LENGTH(t.token) >= 3
      AND t.token <> ''
),
true_s2_tokens AS (
    SELECT DISTINCT
        m.source1_entity_id,
        m.candidate_entity_id,
        t.token
    FROM sample m
    JOIN train_s2 s
        ON m.candidate_entity_id = s.entity_id
    CROSS JOIN UNNEST(
        string_split(
            regexp_replace(s.address_norm, '[^a-z0-9]', ' ', 'g'),
            ' '
        )
    ) AS t(token)
    WHERE LENGTH(t.token) >= 3
      AND t.token <> ''
),
covered AS (
    SELECT DISTINCT
        t.source1_entity_id,
        t.candidate_entity_id
    FROM true_s2_tokens t
    JOIN read_parquet(?) r
        ON t.token = r.token
)
SELECT
    COUNT(*) AS sample_pairs,
    COUNT(c.candidate_entity_id) AS covered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        2
    ) AS coverage_pct
FROM sample s
LEFT JOIN covered c
    ON s.source1_entity_id = c.source1_entity_id
   AND s.candidate_entity_id = c.candidate_entity_id
""", [str(S2_RARE_ADDRESS)]).fetchdf()

print(result)

   sample_pairs  covered_pairs  coverage_pct
0        100000          77374         77.37


In [40]:
# Benchmark rare-address-token candidate generation on 50K S1 entities

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_address_tokens_test AS
SELECT DISTINCT
    s.entity_id AS source1_entity_id,
    s.country_norm,
    t.token
FROM (
    SELECT
        entity_id,
        country_norm,
        address_norm
    FROM train_s1
    ORDER BY entity_id
    LIMIT 50000
) s
CROSS JOIN UNNEST(
    string_split(
        regexp_replace(s.address_norm, '[^a-z0-9]', ' ', 'g'),
        ' '
    )
) AS t(token)
WHERE LENGTH(t.token) >= 3
  AND t.token <> ''
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE address_candidates_test AS
SELECT DISTINCT
    s1.source1_entity_id,
    s2.candidate_entity_id
FROM s1_address_tokens_test s1
JOIN read_parquet(?) s2
    ON s1.country_norm = s2.country_norm
   AND s1.token = s2.token
""", [str(S2_RARE_ADDRESS)])

print(
    con.execute("""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM address_candidates_test
    """).fetchdf()
)

   candidate_pairs  s1_with_candidates  avg_candidates_per_s1
0           436481               20010                  21.81


In [41]:
# Measure rare-address-token blocker recall on the 50K S1 benchmark

result = con.execute("""
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        4
    ) AS recall_pct
FROM gt_s2 g
JOIN (
    SELECT DISTINCT source1_entity_id
    FROM s1_address_tokens_test
) b
    ON g.source1_entity_id = b.source1_entity_id
LEFT JOIN address_candidates_test c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0     83431            28489     34.1468


In [43]:
# Incremental true-pair recovery from rare-address blocking
# over COMBINED_V2 for the 50K S1 benchmark.

result = con.execute("""
WITH address_true AS (
    SELECT DISTINCT
        a.source1_entity_id,
        a.candidate_entity_id
    FROM address_candidates_test a
    JOIN gt_s2 g
        ON a.source1_entity_id = g.source1_entity_id
       AND a.candidate_entity_id = g.candidate_entity_id
)
SELECT
    COUNT(*) AS address_recovered,
    COUNT(*) FILTER (
        WHERE c.candidate_entity_id IS NULL
    ) AS genuinely_new,
    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE c.candidate_entity_id IS NULL
        ) / COUNT(*),
        2
    ) AS new_pct
FROM address_true a
LEFT JOIN read_parquet(?) c
    ON a.source1_entity_id = c.source1_entity_id
   AND a.candidate_entity_id = c.candidate_entity_id
""", [str(COMBINED_V2)]).fetchdf()

print(result)

   address_recovered  genuinely_new  new_pct
0              28489          14640    51.39


In [44]:
# Prepare full S1 rare-address-token postings

con.execute("""
CREATE OR REPLACE VIEW s1_address_tokens AS
SELECT DISTINCT
    s.entity_id AS source1_entity_id,
    s.country_norm,
    t.token
FROM train_s1 s
CROSS JOIN UNNEST(
    string_split(
        regexp_replace(s.address_norm, '[^a-z0-9]', ' ', 'g'),
        ' '
    )
) AS t(token)
WHERE LENGTH(t.token) >= 3
  AND t.token <> ''
""")

s1_count = con.execute("""
    SELECT COUNT(DISTINCT source1_entity_id)
    FROM s1_address_tokens
""").fetchone()[0]

print("S1 entities:", s1_count)
print("Batches:", (s1_count + 24999) // 25000)

S1 entities: 2206821
Batches: 89


In [45]:
# Generate rare-address candidates for batch 0

ADDRESS_DIR = CANDIDATE_DIR / "rare_address_parts"
ADDRESS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 25_000
batch_id = 0

output_path = ADDRESS_DIR / f"part_{batch_id:03d}.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        s1.source1_entity_id,
        s2.candidate_entity_id
    FROM (
        SELECT DISTINCT source1_entity_id, country_norm, token
        FROM s1_address_tokens
        ORDER BY source1_entity_id
        LIMIT {BATCH_SIZE}
    ) s1
    JOIN read_parquet('{S2_RARE_ADDRESS}') s2
      ON s1.country_norm = s2.country_norm
     AND s1.token = s2.token
) TO '{output_path}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Written:", output_path)

print(
    con.execute(f"""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM read_parquet('{output_path}')
    """).fetchdf()
)

Written: D:\business_entity_resolution\dataset\work\candidates\rare_address_parts\part_000.parquet
   candidate_pairs  s1_with_candidates  avg_candidates_per_s1
0            33119                1561                  21.22


In [46]:
# Recall of rare-address blocking on batch 0

result = con.execute(f"""
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        4
    ) AS recall_pct
FROM gt_s2 g
JOIN (
    SELECT DISTINCT source1_entity_id
    FROM s1_address_tokens
    ORDER BY source1_entity_id
    LIMIT {BATCH_SIZE}
) b
    ON g.source1_entity_id = b.source1_entity_id
LEFT JOIN read_parquet('{output_path}') c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0     41605             2316      5.5666


In [47]:
result = con.execute("""
SELECT
    COUNT(*) AS missed_pairs,
    COUNT(*) FILTER (
        WHERE name_jw >= 0.90
          AND address_jw >= 0.75
    ) AS high_both,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE name_jw >= 0.90
              AND address_jw >= 0.75
        ) / COUNT(*),
        2
    ) AS pct
FROM read_parquet(
    'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
)
""").fetchdf()

print(result)

   missed_pairs  high_both    pct
0       1877384     698593  37.21


In [48]:
# Test a composite name + address blocking key on the missed true pairs

result = con.execute("""
SELECT
    COUNT(*) AS missed_pairs,

    COUNT(*) FILTER (
        WHERE
            s1.country_norm = s2.country_norm
            AND LEFT(
                regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 4
            )
            =
            LEFT(
                regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 4
            )
            AND LEFT(
                regexp_replace(s1.address_norm, '[^a-z0-9]', '', 'g'), 4
            )
            =
            LEFT(
                regexp_replace(s2.address_norm, '[^a-z0-9]', '', 'g'), 4
            )
    ) AS recovered_pairs

FROM read_parquet(
    'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
) m
JOIN train_s1 s1
    ON m.source1_entity_id = s1.entity_id
JOIN train_s2 s2
    ON m.candidate_entity_id = s2.entity_id
""").fetchdf()

result["recall_pct"] = (
    100 * result["recovered_pairs"] / result["missed_pairs"]
)

print(result)

   missed_pairs  recovered_pairs  recall_pct
0       1877384           594735   31.678921


In [49]:
# Check genuinely new true pairs from composite name+address blocking

result = con.execute("""
WITH composite AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id
    FROM read_parquet(
        'D:/business_entity_resolution/dataset/work/candidates/missed_true_pairs.parquet'
    ) m
    JOIN train_s1 s1
        ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
        ON m.candidate_entity_id = s2.entity_id
    WHERE
        s1.country_norm = s2.country_norm
        AND LEFT(
            regexp_replace(s1.name_norm, '[^a-z0-9]', '', 'g'), 4
        )
        =
        LEFT(
            regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 4
        )
        AND LEFT(
            regexp_replace(s1.address_norm, '[^a-z0-9]', '', 'g'), 4
        )
        =
        LEFT(
            regexp_replace(s2.address_norm, '[^a-z0-9]', '', 'g'), 4
        )
)
SELECT
    COUNT(*) AS composite_recovered,
    COUNT(*) FILTER (
        WHERE c.candidate_entity_id IS NULL
    ) AS genuinely_new
FROM composite x
LEFT JOIN read_parquet(?) c
    ON x.source1_entity_id = c.source1_entity_id
   AND x.candidate_entity_id = c.candidate_entity_id
""", [str(COMBINED_V2)]).fetchdf()

print(result)

   composite_recovered  genuinely_new
0               594735         594735


In [50]:
# Benchmark composite name+address blocking on 50K S1 entities

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_composite_test AS
SELECT
    entity_id AS source1_entity_id,
    country_norm,
    LEFT(
        regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4
    ) AS name_key,
    LEFT(
        regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4
    ) AS address_key
FROM train_s1
ORDER BY entity_id
LIMIT 50000
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE composite_candidates_test AS
SELECT DISTINCT
    s1.source1_entity_id,
    s2.entity_id AS candidate_entity_id
FROM s1_composite_test s1
JOIN train_s2 s2
    ON s1.country_norm = s2.country_norm
   AND s1.name_key = LEFT(
        regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 4
   )
   AND s1.address_key = LEFT(
        regexp_replace(s2.address_norm, '[^a-z0-9]', '', 'g'), 4
   )
WHERE s1.name_key <> ''
  AND s1.address_key <> ''
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM composite_candidates_test
    """).fetchdf()
)

   candidate_pairs  s1_with_candidates  avg_candidates_per_s1
0           246151               28396                   8.67


In [51]:
# Recall of composite name+address blocker on the 50K S1 benchmark

result = con.execute("""
SELECT
    COUNT(*) AS gt_pairs,
    COUNT(c.candidate_entity_id) AS recovered_pairs,
    ROUND(
        100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
        4
    ) AS recall_pct
FROM gt_s2 g
JOIN s1_composite_test b
    ON g.source1_entity_id = b.source1_entity_id
LEFT JOIN composite_candidates_test c
    ON g.source1_entity_id = c.source1_entity_id
   AND g.candidate_entity_id = c.candidate_entity_id
""").fetchdf()

print(result)

   gt_pairs  recovered_pairs  recall_pct
0     83431            33206     39.8006


In [52]:
COMPOSITE_DIR = CANDIDATE_DIR / "composite_name_address_parts"
COMPOSITE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 25_000
batch_id = 0

output_path = COMPOSITE_DIR / f"part_{batch_id:03d}.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        s1.source1_entity_id,
        s2.entity_id AS candidate_entity_id
    FROM (
        SELECT
            entity_id AS source1_entity_id,
            country_norm,
            LEFT(
                regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4
            ) AS name_key,
            LEFT(
                regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4
            ) AS address_key
        FROM train_s1
        ORDER BY entity_id
        LIMIT {BATCH_SIZE}
    ) s1
    JOIN train_s2 s2
      ON s1.country_norm = s2.country_norm
     AND s1.name_key = LEFT(
            regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 4
         )
     AND s1.address_key = LEFT(
            regexp_replace(s2.address_norm, '[^a-z0-9]', '', 'g'), 4
         )
    WHERE s1.name_key <> ''
      AND s1.address_key <> ''
) TO '{output_path}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Written:", output_path)

print(
    con.execute(f"""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_with_candidates,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM read_parquet('{output_path}')
    """).fetchdf()
)

Written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_000.parquet
   candidate_pairs  s1_with_candidates  avg_candidates_per_s1
0           126557               14150                   8.94


In [53]:
# Generate remaining composite name+address candidate batches

BATCH_SIZE = 25_000
TOTAL_S1 = 2_206_821
NUM_BATCHES = (TOTAL_S1 + BATCH_SIZE - 1) // BATCH_SIZE

for batch_id in range(1, NUM_BATCHES):
    offset = batch_id * BATCH_SIZE
    output_path = COMPOSITE_DIR / f"part_{batch_id:03d}.parquet"

    print(f"Starting batch {batch_id}/{NUM_BATCHES - 1}...")

    con.execute(f"""
    COPY (
        SELECT DISTINCT
            s1.source1_entity_id,
            s2.entity_id AS candidate_entity_id
        FROM (
            SELECT
                entity_id AS source1_entity_id,
                country_norm,
                LEFT(
                    regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4
                ) AS name_key,
                LEFT(
                    regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4
                ) AS address_key
            FROM train_s1
            ORDER BY entity_id
            LIMIT {BATCH_SIZE}
            OFFSET {offset}
        ) s1
        JOIN train_s2 s2
          ON s1.country_norm = s2.country_norm
         AND s1.name_key = LEFT(
                regexp_replace(s2.name_norm, '[^a-z0-9]', '', 'g'), 4
             )
         AND s1.address_key = LEFT(
                regexp_replace(s2.address_norm, '[^a-z0-9]', '', 'g'), 4
             )
        WHERE s1.name_key <> ''
          AND s1.address_key <> ''
    ) TO '{output_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print(f"  written: {output_path}")

print("All composite batches completed.")

Starting batch 1/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_001.parquet
Starting batch 2/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_002.parquet
Starting batch 3/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_003.parquet
Starting batch 4/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_004.parquet
Starting batch 5/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_005.parquet
Starting batch 6/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_006.parquet
Starting batch 7/88...
  written: D:\business_entity_resolution\dataset\work\candidates\composite_name_address_parts\part_007.parquet
Starting batch 8/88...
  written: D:\business_entity_resolutio

In [54]:
# Combine COMBINED_V2 + all composite name/address candidate batches

COMPOSITE_GLOB = str(COMPOSITE_DIR / "part_*.parquet")

COMBINED_V3 = CANDIDATE_DIR / "combined_v3_name_address.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{COMBINED_V2}')

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_parquet('{COMPOSITE_GLOB}')
    )
) TO '{COMBINED_V3}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", COMBINED_V3)

Saved: D:\business_entity_resolution\dataset\work\candidates\combined_v3_name_address.parquet


In [55]:
# Measure candidate count, S1 coverage, and ground-truth recall

print(
    con.execute(f"""
        SELECT
            COUNT(*) AS candidate_pairs,
            COUNT(DISTINCT source1_entity_id) AS s1_covered,
            ROUND(
                COUNT(*) * 1.0 /
                COUNT(DISTINCT source1_entity_id), 2
            ) AS avg_candidates_per_s1
        FROM read_parquet('{COMBINED_V3}')
    """).fetchdf()
)

print(
    con.execute(f"""
        SELECT
            COUNT(*) AS gt_pairs,
            COUNT(c.candidate_entity_id) AS recovered_pairs,
            ROUND(
                100.0 * COUNT(c.candidate_entity_id) / COUNT(*),
                4
            ) AS recall_pct
        FROM gt_s2 g
        LEFT JOIN read_parquet('{COMBINED_V3}') c
          ON g.source1_entity_id = c.source1_entity_id
         AND g.candidate_entity_id = c.candidate_entity_id
    """).fetchdf()
)

   candidate_pairs  s1_covered  avg_candidates_per_s1
0         31024695     1951033                   15.9
   gt_pairs  recovered_pairs  recall_pct
0   3693619          2410970     65.2739


In [56]:
# Score all COMBINED_V3 candidates

V3_SCORED = CANDIDATE_DIR / "combined_v3_scored.parquet"

con.execute(f"""
COPY (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,

        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ) AS name_jw,

        levenshtein(
            s1.name_norm,
            s2.name_norm
        ) AS name_edit_distance,

        jaro_winkler_similarity(
            s1.address_norm,
            s2.address_norm
        ) AS address_jw

    FROM read_parquet('{COMBINED_V3}') c

    JOIN train_s1 s1
      ON c.source1_entity_id = s1.entity_id

    JOIN train_s2 s2
      ON c.candidate_entity_id = s2.entity_id
) TO '{V3_SCORED}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", V3_SCORED)


Saved: D:\business_entity_resolution\dataset\work\candidates\combined_v3_scored.parquet


In [57]:
# Label V3 scored candidates against ground truth

V3_LABELED = CANDIDATE_DIR / "combined_v3_scored_labeled.parquet"

con.execute(f"""
COPY (
    SELECT
        c.*,
        CASE
            WHEN g.candidate_entity_id IS NOT NULL THEN 1
            ELSE 0
        END AS is_match,

        0.35 * c.name_jw
        + 0.65 * c.address_jw AS combined_score

    FROM read_parquet('{V3_SCORED}') c

    LEFT JOIN gt_s2 g
      ON c.source1_entity_id = g.source1_entity_id
     AND c.candidate_entity_id = g.candidate_entity_id
) TO '{V3_LABELED}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", V3_LABELED)

Saved: D:\business_entity_resolution\dataset\work\candidates\combined_v3_scored_labeled.parquet


In [58]:
# Compare score distributions for true vs non-matching candidates

print(
    con.execute(f"""
        SELECT
            is_match,
            COUNT(*) AS candidates,
            ROUND(AVG(combined_score), 4) AS avg_score,
            ROUND(MEDIAN(combined_score), 4) AS median_score,
            ROUND(
                APPROX_QUANTILE(combined_score, 0.10), 4
            ) AS p10,
            ROUND(
                APPROX_QUANTILE(combined_score, 0.25), 4
            ) AS p25,
            ROUND(
                APPROX_QUANTILE(combined_score, 0.75), 4
            ) AS p75,
            ROUND(
                APPROX_QUANTILE(combined_score, 0.90), 4
            ) AS p90
        FROM read_parquet('{V3_LABELED}')
        GROUP BY is_match
        ORDER BY is_match DESC
    """).fetchdf()
)

   is_match  candidates  avg_score  median_score     p10     p25     p75  \
0         1     2410970     0.8853        0.9256  0.7744  0.8528  0.9628   
1         0    28613725     0.6984        0.7070  0.6302  0.6694  0.7386   

      p90  
0  0.9849  
1  0.7698  


In [59]:
threshold_sweep = con.execute(f"""
SELECT
    threshold,
    tp,
    fp,
    fn,
    ROUND(tp::DOUBLE / NULLIF(tp + fp, 0), 4) AS precision,
    ROUND(tp::DOUBLE / NULLIF(tp + fn, 0), 4) AS recall,
    ROUND(
        1.25 * tp::DOUBLE /
        NULLIF(1.25 * tp + 0.25 * fn + fp, 0),
        4
    ) AS f05
FROM (
    SELECT
        threshold,
        SUM(CASE WHEN is_match = 1 AND combined_score >= threshold THEN 1 ELSE 0 END) AS tp,
        SUM(CASE WHEN is_match = 0 AND combined_score >= threshold THEN 1 ELSE 0 END) AS fp,
        SUM(CASE WHEN is_match = 1 AND combined_score < threshold THEN 1 ELSE 0 END) AS fn
    FROM read_parquet('{V3_LABELED}')
    CROSS JOIN (
        SELECT * FROM range(70, 96, 1)
    ) t(i)
    CROSS JOIN LATERAL (
        SELECT i / 100.0 AS threshold
    )
    GROUP BY threshold
)
ORDER BY f05 DESC
""").df()

threshold_sweep

,threshold,tp,fp,fn,precision,recall,f05
0,0.87,1681315.0,203146.0,729655.0,0.8922,0.6974,0.8450
1,0.88,1613948.0,175720.0,797022.0,0.9018,0.6694,0.8433
2,0.89,1550366.0,157326.0,860604.0,0.9079,0.6430,0.8388
3,0.86,1754206.0,268843.0,656764.0,0.8671,0.7276,0.8351
4,0.90,1480912.0,140737.0,930058.0,0.9132,0.6142,0.8322
5,0.91,1393894.0,123672.0,1017076.0,0.9185,0.5781,0.8218
6,0.92,1280613.0,105856.0,1130357.0,0.9237,0.5312,0.8047
7,0.85,1828029.0,439606.0,582941.0,0.8061,0.7582,0.7961
8,0.93,1141377.0,87655.0,1269593.0,0.9287,0.4734,0.7789
9,0.94,986859.0,69245.0,1424111.0,0.9344,0.4093,0.7436


In [60]:
missed_v3 = con.execute(f"""
SELECT
    COUNT(*) AS missed_pairs,
    ROUND(AVG(jaro_winkler_similarity(s1.name_norm, s2.name_norm)), 4) AS avg_name_jw,
    ROUND(AVG(jaro_winkler_similarity(s1.address_norm, s2.address_norm)), 4) AS avg_address_jw,
    ROUND(AVG(
        0.35 * jaro_winkler_similarity(s1.name_norm, s2.name_norm)
        + 0.65 * jaro_winkler_similarity(s1.address_norm, s2.address_norm)
    ), 4) AS avg_combined_score,
    ROUND(
        100.0 * AVG(
            CASE WHEN jaro_winkler_similarity(s1.name_norm, s2.name_norm) >= 0.90
                 THEN 1 ELSE 0 END
        ), 2
    ) AS name_ge_090_pct,
    ROUND(
        100.0 * AVG(
            CASE WHEN jaro_winkler_similarity(s1.address_norm, s2.address_norm) >= 0.70
                 THEN 1 ELSE 0 END
        ), 2
    ) AS address_ge_070_pct
FROM read_parquet('{WORK_DIR / "candidates" / "missed_true_pairs.parquet"}') m
JOIN train_s1 s1
  ON m.source1_entity_id = s1.entity_id
JOIN train_s2 s2
  ON m.candidate_entity_id = s2.entity_id
""").df()

missed_v3

,missed_pairs,avg_name_jw,avg_address_jw,avg_combined_score,name_ge_090_pct,address_ge_070_pct
0,1877384,0.7866,0.8194,0.8079,47.82,82.38


In [62]:
addr_overlap = con.execute(f"""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id
    FROM read_parquet('{WORK_DIR / "candidates" / "missed_true_pairs.parquet"}') m
),
pairs AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        s1.address_norm AS s1_address,
        s2.address_norm AS s2_address
    FROM missed m
    JOIN train_s1 s1
      ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
      ON m.candidate_entity_id = s2.entity_id
),
overlap AS (
    SELECT
        *,
        list_intersect(
            regexp_split_to_array(s1_address, '\\s+'),
            regexp_split_to_array(s2_address, '\\s+')
        ) AS shared_tokens
    FROM pairs
)
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN len(shared_tokens) >= 1 THEN 1 ELSE 0 END) AS shared_1plus,
    SUM(CASE WHEN len(shared_tokens) >= 2 THEN 1 ELSE 0 END) AS shared_2plus,
    SUM(CASE WHEN len(shared_tokens) >= 3 THEN 1 ELSE 0 END) AS shared_3plus
FROM overlap
""").df()

addr_overlap

,total,shared_1plus,shared_2plus,shared_3plus
0,1877384,1809388.0,1808264.0,1782840.0


In [63]:
addr_pair_stats = con.execute("""
WITH s2_tokens AS (
    SELECT
        entity_id,
        country_norm,
        regexp_split_to_array(
            regexp_replace(address_norm, '\\s+', ' '),
            ' '
        ) AS tokens
    FROM train_s2
    WHERE address_norm <> ''
),
pairs AS (
    SELECT
        entity_id,
        country_norm,
        tokens[i] || '|' || tokens[i + 1] AS token_pair
    FROM s2_tokens,
         generate_series(1, array_length(tokens) - 1) AS t(i)
    WHERE length(tokens[i]) >= 2
      AND length(tokens[i + 1]) >= 2
),
freq AS (
    SELECT
        country_norm,
        token_pair,
        COUNT(*) AS posting_count
    FROM pairs
    GROUP BY country_norm, token_pair
)
SELECT
    COUNT(*) AS distinct_pairs,
    SUM(CASE WHEN posting_count <= 5 THEN 1 ELSE 0 END) AS pairs_le_5,
    SUM(CASE WHEN posting_count <= 10 THEN 1 ELSE 0 END) AS pairs_le_10,
    SUM(CASE WHEN posting_count <= 25 THEN 1 ELSE 0 END) AS pairs_le_25,
    SUM(CASE WHEN posting_count <= 50 THEN 1 ELSE 0 END) AS pairs_le_50,
    ROUND(AVG(posting_count), 2) AS avg_posting,
    ROUND(MEDIAN(posting_count), 2) AS median_posting,
    ROUND(APPROX_QUANTILE(posting_count, 0.90), 2) AS p90_posting,
    ROUND(APPROX_QUANTILE(posting_count, 0.95), 2) AS p95_posting,
    MAX(posting_count) AS max_posting
FROM freq
""").df()

addr_pair_stats

,distinct_pairs,pairs_le_5,pairs_le_10,pairs_le_25,pairs_le_50,avg_posting,median_posting,p90_posting,p95_posting,max_posting
0,5315660,4794060.0,5057009.0,5211955.0,5264247.0,5.52,1.0,5,10,148860


In [64]:
adj_pair_coverage = con.execute(f"""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id
    FROM read_parquet('{WORK_DIR / "candidates" / "missed_true_pairs.parquet"}') m
),
pairs AS (
    SELECT
        m.source1_entity_id,
        m.candidate_entity_id,
        s1.address_norm AS s1_address,
        s2.address_norm AS s2_address
    FROM missed m
    JOIN train_s1 s1
      ON m.source1_entity_id = s1.entity_id
    JOIN train_s2 s2
      ON m.candidate_entity_id = s2.entity_id
),
tokenized AS (
    SELECT
        *,
        regexp_split_to_array(
            regexp_replace(s1_address, '\\s+', ' '), ' '
        ) AS s1_tokens,
        regexp_split_to_array(
            regexp_replace(s2_address, '\\s+', ' '), ' '
        ) AS s2_tokens
    FROM pairs
),
with_pairs AS (
    SELECT
        *,
        list_intersect(
            list_transform(
                generate_series(1, array_length(s1_tokens) - 1),
                i -> s1_tokens[i] || '|' || s1_tokens[i + 1]
            ),
            list_transform(
                generate_series(1, array_length(s2_tokens) - 1),
                i -> s2_tokens[i] || '|' || s2_tokens[i + 1]
            )
        ) AS shared_pairs
    FROM tokenized
)
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN len(shared_pairs) >= 1 THEN 1 ELSE 0 END) AS shared_pair_1plus,
    SUM(CASE WHEN len(shared_pairs) >= 2 THEN 1 ELSE 0 END) AS shared_pair_2plus
FROM with_pairs
""").df()

adj_pair_coverage

,total,shared_pair_1plus,shared_pair_2plus
0,1877384,1743794.0,1507324.0


In [66]:
from pathlib import Path

for p in DATA_ROOT.rglob("*"):
    if p.is_file() and "test" in p.name.lower():
        print(p)

D:\business_entity_resolution\dataset\test\test_source1.tsv
D:\business_entity_resolution\dataset\test\test_source2.tsv
D:\business_entity_resolution\dataset\test\test_source3.tsv


In [68]:
TEST_S1 = DATA_ROOT  / "test" / "test_source1.tsv"
TEST_S2 = DATA_ROOT  / "test" / "test_source2.tsv"
TEST_S3 = DATA_ROOT  / "test" / "test_source3.tsv"

for name, path in [
    ("TEST S1", TEST_S1),
    ("TEST S2", TEST_S2),
    ("TEST S3", TEST_S3),
]:
    print(f"\n{name}: {path}")
    print(con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT entity_id) AS unique_ids,
            COUNT(DISTINCT country) AS countries
        FROM read_csv_auto(
            '{path}',
            sep='\\t',
            header=true
        )
    """).df())

    print(con.execute(f"""
        SELECT *
        FROM read_csv_auto(
            '{path}',
            sep='\\t',
            header=true
        )
        LIMIT 3
    """).df())


TEST S1: D:\business_entity_resolution\dataset\test\test_source1.tsv
      rows  unique_ids  countries
0  1732544     1732544          3
      entity_id         business_name  \
0  S1-714132312       Zephay Labs Inc   
1  S1-106407869  Vision Partners Corp   
2  S1-156285671         << Team Ecole   

                                    business_address country  
0                        2621 Cotten Road, Tyler, TX      US  
1             IA, Iowa City, 1064 Newton Rd, Unit 11      US  
2  175 Boulevard du Président Franklin Roosevelt,...  France  

TEST S2: D:\business_entity_resolution\dataset\test\test_source2.tsv
      rows  unique_ids  countries
0  4887273     4887273          3
      entity_id                                   business_name  \
0  S2-582902882                          The Hopkins Blue Oneim   
1  S2-919947601  IW Trading Limited South West  Delhi Public of   
2   S2-36166996                                  Collège Joseph   

                                    bu

In [69]:
TEST_NORM_DIR = DATA_ROOT / "work" / "normalized_test"
TEST_NORM_DIR.mkdir(parents=True, exist_ok=True)

TEST_NORM_S1 = TEST_NORM_DIR / "test_source1.parquet"
TEST_NORM_S2 = TEST_NORM_DIR / "test_source2.parquet"
TEST_NORM_S3 = TEST_NORM_DIR / "test_source3.parquet"

def normalize_test_file(src, dst):
    (
        pl.scan_csv(
            src,
            separator="\t",
            has_header=True,
            infer_schema_length=10000,
        )
        .with_columns([
            pl.col("business_name")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[\p{P}\p{S}]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("name_norm"),

            pl.col("business_address")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[\p{P}\p{S}]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("address_norm"),

            pl.col("country")
                .fill_null("")
                .str.normalize("NFKC")
                .str.strip_chars()
                .alias("country_norm"),
        ])
        .collect(streaming=True)
        .write_parquet(dst, compression="zstd")
    )

for src, dst in [
    (TEST_S1, TEST_NORM_S1),
    (TEST_S2, TEST_NORM_S2),
    (TEST_S3, TEST_NORM_S3),
]:
    if not dst.exists():
        print(f"Normalizing {src.name}...")
        normalize_test_file(src, dst)
        print(f"Written: {dst}")
    else:
        print(f"Already exists: {dst}")

Normalizing test_source1.tsv...


C:\Users\KRISH\tmp\ipykernel_16832\78246166.py:41: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


Written: D:\business_entity_resolution\dataset\work\normalized_test\test_source1.parquet
Normalizing test_source2.tsv...
Written: D:\business_entity_resolution\dataset\work\normalized_test\test_source2.parquet
Normalizing test_source3.tsv...
Written: D:\business_entity_resolution\dataset\work\normalized_test\test_source3.parquet


In [71]:
con.execute(f"""
CREATE OR REPLACE VIEW test_s1 AS
SELECT * FROM read_parquet('{TEST_NORM_S1}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW test_s2 AS
SELECT * FROM read_parquet('{TEST_NORM_S2}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW test_s3 AS
SELECT * FROM read_parquet('{TEST_NORM_S3}');
""")

print("Test views created.")

Test views created.


In [72]:
for table in ["test_s1", "test_s2", "test_s3"]:
    print(table, con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])

test_s1 1732544
test_s2 4887273
test_s3 5082316


In [73]:
TEST_CANDIDATE_DIR = WORK_DIR / "test_candidates"
TEST_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

print(TEST_CANDIDATE_DIR)

D:\business_entity_resolution\dataset\work\test_candidates


In [74]:
con.execute("""
CREATE OR REPLACE VIEW test_s1_block AS
SELECT
    entity_id AS source1_entity_id,
    country_norm,
    name_norm,
    address_norm,
    country_norm
        || '|'
        || LEFT(regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4)
        || '|'
        || LEFT(regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4)
        AS block_key
FROM test_s1;
""")

con.execute("""
CREATE OR REPLACE VIEW test_s2_block AS
SELECT
    entity_id AS candidate_entity_id,
    country_norm,
    name_norm,
    address_norm,
    country_norm
        || '|'
        || LEFT(regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4)
        || '|'
        || LEFT(regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4)
        AS block_key
FROM test_s2;
""")

print("Test blocker views created.")

Test blocker views created.


In [75]:
TEST_S2_CAND_DIR = TEST_CANDIDATE_DIR / "s2_parts"
TEST_S2_CAND_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 25_000

s1_count = con.execute("SELECT COUNT(*) FROM test_s1").fetchone()[0]
num_batches = (s1_count + BATCH_SIZE - 1) // BATCH_SIZE

print(f"S1 rows: {s1_count:,}")
print(f"Batch size: {BATCH_SIZE:,}")
print(f"Total batches: {num_batches}")

S1 rows: 1,732,544
Batch size: 25,000
Total batches: 70


In [76]:
for batch_id in range(num_batches):
    out = TEST_S2_CAND_DIR / f"part_{batch_id:03d}.parquet"

    if out.exists():
        print(f"[{batch_id+1}/{num_batches}] exists, skipping")
        continue

    offset = batch_id * BATCH_SIZE

    print(f"[{batch_id+1}/{num_batches}] generating...")

    con.execute(f"""
    COPY (
        SELECT DISTINCT
            s1.source1_entity_id,
            s2.candidate_entity_id
        FROM (
            SELECT *
            FROM test_s1_block
            ORDER BY source1_entity_id
            LIMIT {BATCH_SIZE}
            OFFSET {offset}
        ) s1
        JOIN test_s2_block s2
          ON s1.block_key = s2.block_key
    )
    TO '{out}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

    n = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{out}')"
    ).fetchone()[0]

    print(f"    candidates: {n:,}")

[1/70] generating...
    candidates: 209,840
[2/70] generating...
    candidates: 209,521
[3/70] generating...
    candidates: 224,203
[4/70] generating...
    candidates: 203,608
[5/70] generating...
    candidates: 219,240
[6/70] generating...
    candidates: 214,285
[7/70] generating...
    candidates: 213,219
[8/70] generating...
    candidates: 207,218
[9/70] generating...
    candidates: 211,090
[10/70] generating...
    candidates: 202,288
[11/70] generating...
    candidates: 209,063
[12/70] generating...
    candidates: 205,831
[13/70] generating...
    candidates: 215,856
[14/70] generating...
    candidates: 219,982
[15/70] generating...
    candidates: 201,733
[16/70] generating...
    candidates: 216,289
[17/70] generating...
    candidates: 217,049
[18/70] generating...
    candidates: 217,631
[19/70] generating...
    candidates: 210,963
[20/70] generating...
    candidates: 222,626
[21/70] generating...
    candidates: 207,607
[22/70] generating...
    candidates: 214,2

In [77]:
TEST_S3_CAND_DIR = TEST_CANDIDATE_DIR / "s3_parts"
TEST_S3_CAND_DIR.mkdir(parents=True, exist_ok=True)

con.execute("""
CREATE OR REPLACE VIEW test_s3_block AS
SELECT
    entity_id AS candidate_entity_id,
    country_norm,
    name_norm,
    address_norm,
    country_norm
        || '|'
        || LEFT(regexp_replace(name_norm, '[^a-z0-9]', '', 'g'), 4)
        || '|'
        || LEFT(regexp_replace(address_norm, '[^a-z0-9]', '', 'g'), 4)
        AS block_key
FROM test_s3;
""")

print("S3 blocker created.")

S3 blocker created.


In [78]:
for batch_id in range(num_batches):
    out = TEST_S3_CAND_DIR / f"part_{batch_id:03d}.parquet"

    if out.exists():
        print(f"[{batch_id+1}/{num_batches}] exists, skipping")
        continue

    offset = batch_id * BATCH_SIZE

    print(f"[{batch_id+1}/{num_batches}] generating...")

    con.execute(f"""
    COPY (
        SELECT DISTINCT
            s1.source1_entity_id,
            s2.candidate_entity_id
        FROM (
            SELECT *
            FROM test_s1_block
            ORDER BY source1_entity_id
            LIMIT {BATCH_SIZE}
            OFFSET {offset}
        ) s1
        JOIN test_s3_block s2
          ON s1.block_key = s2.block_key
    )
    TO '{out}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

    n = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{out}')"
    ).fetchone()[0]

    print(f"    candidates: {n:,}")

[1/70] generating...
    candidates: 247,305
[2/70] generating...
    candidates: 241,656
[3/70] generating...
    candidates: 248,078
[4/70] generating...
    candidates: 237,884
[5/70] generating...
    candidates: 244,753
[6/70] generating...
    candidates: 235,663
[7/70] generating...
    candidates: 238,577
[8/70] generating...
    candidates: 233,692
[9/70] generating...
    candidates: 233,869
[10/70] generating...
    candidates: 228,236
[11/70] generating...
    candidates: 241,210
[12/70] generating...
    candidates: 228,858
[13/70] generating...
    candidates: 237,548
[14/70] generating...
    candidates: 247,798
[15/70] generating...
    candidates: 230,156
[16/70] generating...
    candidates: 234,623
[17/70] generating...
    candidates: 242,157
[18/70] generating...
    candidates: 242,935
[19/70] generating...
    candidates: 237,271
[20/70] generating...
    candidates: 244,965
[21/70] generating...
    candidates: 239,607
[22/70] generating...
    candidates: 240,3

In [79]:
TEST_S2_CAND = TEST_CANDIDATE_DIR / "test_s2_candidates.parquet"
TEST_S3_CAND = TEST_CANDIDATE_DIR / "test_s3_candidates.parquet"

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet('{TEST_S2_CAND_DIR / "part_*.parquet"}')
)
TO '{TEST_S2_CAND}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet('{TEST_S3_CAND_DIR / "part_*.parquet"}')
)
TO '{TEST_S3_CAND}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("S2 candidates:",
      con.execute(f"SELECT COUNT(*) FROM read_parquet('{TEST_S2_CAND}')").fetchone()[0])

print("S3 candidates:",
      con.execute(f"SELECT COUNT(*) FROM read_parquet('{TEST_S3_CAND}')").fetchone()[0])

S2 candidates: 14691698
S3 candidates: 16540551


In [80]:
for path, label in [
    (TEST_S2_CAND, "S2"),
    (TEST_S3_CAND, "S3")
]:
    print(f"\n{label}")
    print(con.execute(f"""
        SELECT
            COUNT(*) AS candidates,
            COUNT(DISTINCT source1_entity_id) AS covered_s1,
            ROUND(AVG(cnt), 2) AS avg_candidates,
            MEDIAN(cnt) AS median_candidates,
            APPROX_QUANTILE(cnt, 0.95) AS p95_candidates,
            MAX(cnt) AS max_candidates
        FROM (
            SELECT source1_entity_id, COUNT(*) AS cnt
            FROM read_parquet('{path}')
            GROUP BY source1_entity_id
        )
    """).df())


S2
   candidates  covered_s1  avg_candidates  median_candidates  p95_candidates  \
0     1018194     1018194           14.43                2.0              66   

   max_candidates  
0            1870  

S3
   candidates  covered_s1  avg_candidates  median_candidates  p95_candidates  \
0     1041251     1041251           15.89                2.0              74   

   max_candidates  
0            2348  


In [81]:
TEST_S2_SCORED = TEST_CANDIDATE_DIR / "test_s2_scored.parquet"

con.execute(f"""
COPY (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,
        jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        ) AS name_jw,
        levenshtein(
            s1.name_norm,
            s2.name_norm
        ) AS name_edit_distance,
        jaro_winkler_similarity(
            s1.address_norm,
            s2.address_norm
        ) AS address_jw,
        0.35 * jaro_winkler_similarity(
            s1.name_norm,
            s2.name_norm
        )
        +
        0.65 * jaro_winkler_similarity(
            s1.address_norm,
            s2.address_norm
        ) AS combined_score
    FROM read_parquet('{TEST_S2_CAND}') c
    JOIN test_s1 s1
      ON c.source1_entity_id = s1.entity_id
    JOIN test_s2 s2
      ON c.candidate_entity_id = s2.entity_id
)
TO '{TEST_S2_SCORED}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("S2 scoring complete.")
print(
    con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{TEST_S2_SCORED}')"
    ).fetchone()[0]
)

S2 scoring complete.
14691698


In [82]:
print("Candidate rows:",
      con.execute(f"""
          SELECT COUNT(*)
          FROM read_parquet('{TEST_S2_CAND}')
      """).fetchone()[0])

print("Distinct candidate pairs:",
      con.execute(f"""
          SELECT COUNT(DISTINCT source1_entity_id || '|' || candidate_entity_id)
          FROM read_parquet('{TEST_S2_CAND}')
      """).fetchone()[0])

print("Test S1 rows / distinct IDs:",
      con.execute("""
          SELECT COUNT(*), COUNT(DISTINCT entity_id)
          FROM test_s1
      """).fetchone())

print("Test S2 rows / distinct IDs:",
      con.execute("""
          SELECT COUNT(*), COUNT(DISTINCT entity_id)
          FROM test_s2
      """).fetchone())

Candidate rows: 14691698
Distinct candidate pairs: 14691698
Test S1 rows / distinct IDs: (1732544, 1732544)
Test S2 rows / distinct IDs: (4887273, 4887273)


In [83]:
print(con.execute(f"""
SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT c.source1_entity_id || '|' || c.candidate_entity_id)
        AS distinct_pairs
FROM read_parquet('{TEST_S2_CAND}') c
JOIN test_s1 s1
  ON c.source1_entity_id = s1.entity_id
JOIN test_s2 s2
  ON c.candidate_entity_id = s2.entity_id
""").df())

   joined_rows  distinct_pairs
0     14691698        14691698


In [84]:
con.execute(f"""
SELECT
    COUNT(*) AS candidates,
    ROUND(AVG(combined_score), 4) AS avg_score,
    ROUND(MEDIAN(combined_score), 4) AS median_score,
    ROUND(APPROX_QUANTILE(combined_score, 0.50), 4) AS p50,
    ROUND(APPROX_QUANTILE(combined_score, 0.75), 4) AS p75,
    ROUND(APPROX_QUANTILE(combined_score, 0.90), 4) AS p90,
    ROUND(APPROX_QUANTILE(combined_score, 0.95), 4) AS p95,
    ROUND(APPROX_QUANTILE(combined_score, 0.99), 4) AS p99,
    SUM(CASE WHEN combined_score >= 0.87 THEN 1 ELSE 0 END) AS ge_087,
    SUM(CASE WHEN combined_score >= 0.90 THEN 1 ELSE 0 END) AS ge_090
FROM read_parquet('{TEST_S2_SCORED}')
""").df()

,candidates,avg_score,median_score,p50,p75,p90,p95,p99,ge_087,ge_090
0,14691698,0.7509,0.7357,0.7356,0.8356,0.8803,0.9375,0.9862,1737979.0,1143525.0


In [85]:
con.execute(f"""
SELECT
    COUNT(*) AS s1_with_matches,
    ROUND(AVG(match_count), 2) AS avg_matches,
    MEDIAN(match_count) AS median_matches,
    APPROX_QUANTILE(match_count, 0.90) AS p90_matches,
    APPROX_QUANTILE(match_count, 0.95) AS p95_matches,
    MAX(match_count) AS max_matches
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM read_parquet('{TEST_S2_SCORED}')
    WHERE combined_score >= 0.87
    GROUP BY source1_entity_id
)
""").df()

,s1_with_matches,avg_matches,median_matches,p90_matches,p95_matches,max_matches
0,794537,2.19,1.0,3,4,265


In [86]:
TEST_S3_SCORED = TEST_CANDIDATE_DIR / "test_s3_scored.parquet"

con.execute(f"""
COPY (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,

        jaro_winkler_similarity(
            s1.name_norm,
            s3.name_norm
        ) AS name_jw,

        levenshtein(
            s1.name_norm,
            s3.name_norm
        ) AS name_edit_distance,

        jaro_winkler_similarity(
            s1.address_norm,
            s3.address_norm
        ) AS address_jw,

        0.35 * jaro_winkler_similarity(
            s1.name_norm,
            s3.name_norm
        )
        +
        0.65 * jaro_winkler_similarity(
            s1.address_norm,
            s3.address_norm
        ) AS combined_score

    FROM read_parquet('{TEST_S3_CAND}') c

    JOIN test_s1 s1
      ON c.source1_entity_id = s1.entity_id

    JOIN test_s3 s3
      ON c.candidate_entity_id = s3.entity_id
)
TO '{TEST_S3_SCORED}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(
    "S3 scored rows:",
    con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{TEST_S3_SCORED}')"
    ).fetchone()[0]
)

S3 scored rows: 16540551


In [87]:
print(con.execute(f"""
SELECT
    COUNT(*) AS candidates,
    ROUND(AVG(combined_score), 4) AS avg_score,
    ROUND(MEDIAN(combined_score), 4) AS median_score,
    ROUND(APPROX_QUANTILE(combined_score, 0.75), 4) AS p75,
    ROUND(APPROX_QUANTILE(combined_score, 0.90), 4) AS p90,
    ROUND(APPROX_QUANTILE(combined_score, 0.95), 4) AS p95,
    ROUND(APPROX_QUANTILE(combined_score, 0.99), 4) AS p99,
    SUM(CASE WHEN combined_score >= 0.87 THEN 1 ELSE 0 END) AS ge_087,
    SUM(CASE WHEN combined_score >= 0.90 THEN 1 ELSE 0 END) AS ge_090
FROM read_parquet('{TEST_S3_SCORED}')
""").df())

print("\nMatches per S1 at threshold 0.87:")

print(con.execute(f"""
SELECT
    COUNT(*) AS s1_with_matches,
    ROUND(AVG(match_count), 2) AS avg_matches,
    MEDIAN(match_count) AS median_matches,
    APPROX_QUANTILE(match_count, 0.90) AS p90_matches,
    APPROX_QUANTILE(match_count, 0.95) AS p95_matches,
    MAX(match_count) AS max_matches
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM read_parquet('{TEST_S3_SCORED}')
    WHERE combined_score >= 0.87
    GROUP BY source1_entity_id
)
""").df())

   candidates  avg_score  median_score     p75     p90     p95     p99  \
0    16540551     0.7279        0.7146  0.7831  0.8693  0.9151  0.9699   

      ge_087     ge_090  
0  1633758.0  1015108.0  

Matches per S1 at threshold 0.87:
   s1_with_matches  avg_matches  median_matches  p90_matches  p95_matches  \
0           758861         2.15             1.0            3            4   

   max_matches  
0          259  


In [88]:
SUBMISSION_DIR = DATA_ROOT / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PAIRS_TSV = SUBMISSION_DIR / "candidate_pairs.tsv"

con.execute(f"""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet('{TEST_S2_SCORED}')

    UNION

    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM read_parquet('{TEST_S3_SCORED}')
)
TO '{CANDIDATE_PAIRS_TSV}'
(
    FORMAT CSV,
    DELIMITER '\\t',
    HEADER TRUE
);
""")

print(f"Written: {CANDIDATE_PAIRS_TSV}")

Written: D:\business_entity_resolution\dataset\submission\candidate_pairs.tsv


In [91]:
CANDIDATE_PAIRS_TSV = SUBMISSION_DIR / "candidate_pairs.tsv"

con.execute(f"""
COPY (
    SELECT
        s1.entity_id AS source1_entity_id,
        COALESCE(
            string_agg(c.candidate_entity_id, ',' ORDER BY c.candidate_entity_id),
            ''
        ) AS candidate_entity_ids
    FROM test_s1 s1
    LEFT JOIN (
        SELECT source1_entity_id, candidate_entity_id
        FROM read_parquet('{TEST_S2_SCORED}')
        UNION
        SELECT source1_entity_id, candidate_entity_id
        FROM read_parquet('{TEST_S3_SCORED}')
    ) c
      ON s1.entity_id = c.source1_entity_id
    GROUP BY s1.entity_id
    ORDER BY s1.entity_id
)
TO '{CANDIDATE_PAIRS_TSV}'
(
    FORMAT CSV,
    DELIMITER '\\t',
    HEADER TRUE
);
""")

print(f"Written: {CANDIDATE_PAIRS_TSV}")

OutOfMemoryException: Out of Memory Error: failed to allocate data of size 32.0 KiB (1.8 GiB/1.8 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [92]:
if CANDIDATE_PAIRS_TSV.exists():
    CANDIDATE_PAIRS_TSV.unlink()

print("Ready:", CANDIDATE_PAIRS_TSV)

Ready: D:\business_entity_resolution\dataset\submission\candidate_pairs.tsv


In [93]:
CAND_BATCH = 50_000

s1_count = con.execute("SELECT COUNT(*) FROM test_s1").fetchone()[0]
num_out_batches = (s1_count + CAND_BATCH - 1) // CAND_BATCH

print(f"S1 rows: {s1_count:,}")
print(f"Output batches: {num_out_batches}")

S1 rows: 1,732,544
Output batches: 35


In [95]:
first_batch = True

for batch_id in range(num_out_batches):
    offset = batch_id * CAND_BATCH

    print(f"[{batch_id + 1}/{num_out_batches}]")

    df = con.execute(f"""
        WITH batch_s1 AS (
            SELECT entity_id
            FROM test_s1
            ORDER BY entity_id
            LIMIT {CAND_BATCH}
            OFFSET {offset}
        ),
        candidates AS (
            SELECT
                source1_entity_id,
                candidate_entity_id
            FROM read_parquet('{TEST_S2_SCORED}')
            WHERE source1_entity_id IN (
                SELECT entity_id FROM batch_s1
            )

            UNION

            SELECT
                source1_entity_id,
                candidate_entity_id
            FROM read_parquet('{TEST_S3_SCORED}')
            WHERE source1_entity_id IN (
                SELECT entity_id FROM batch_s1
            )
        )
        SELECT
            b.entity_id AS source1_entity_id,
            COALESCE(
                string_agg(
                    c.candidate_entity_id,
                    ',' ORDER BY c.candidate_entity_id
                ),
                ''
            ) AS candidate_entity_ids
        FROM batch_s1 b
        LEFT JOIN candidates c
          ON b.entity_id = c.source1_entity_id
        GROUP BY b.entity_id
        ORDER BY b.entity_id
    """).df()

    df.to_csv(
        CANDIDATE_PAIRS_TSV,
        sep="\t",
        index=False,
        header=first_batch,
        mode="w" if first_batch else "a"
    )

    first_batch = False

    print(f"    rows written: {len(df):,}")

[1/35]
    rows written: 50,000
[2/35]
    rows written: 50,000
[3/35]
    rows written: 50,000
[4/35]
    rows written: 50,000
[5/35]
    rows written: 50,000
[6/35]
    rows written: 50,000
[7/35]
    rows written: 50,000
[8/35]
    rows written: 50,000
[9/35]
    rows written: 50,000
[10/35]
    rows written: 50,000
[11/35]
    rows written: 50,000
[12/35]
    rows written: 50,000
[13/35]
    rows written: 50,000
[14/35]
    rows written: 50,000
[15/35]
    rows written: 50,000
[16/35]
    rows written: 50,000
[17/35]
    rows written: 50,000
[18/35]
    rows written: 50,000
[19/35]
    rows written: 50,000
[20/35]
    rows written: 50,000
[21/35]
    rows written: 50,000
[22/35]
    rows written: 50,000
[23/35]
    rows written: 50,000
[24/35]
    rows written: 50,000
[25/35]
    rows written: 50,000
[26/35]
    rows written: 50,000
[27/35]
    rows written: 50,000
[28/35]
    rows written: 50,000
[29/35]
    rows written: 50,000
[30/35]
    rows written: 50,000
[31/35]
    rows wr

In [96]:
print(
    con.execute(f"""
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT source1_entity_id) AS unique_s1
        FROM read_csv(
            '{CANDIDATE_PAIRS_TSV}',
            delim='\\t',
            header=true
        )
    """).df()
)

print("\nColumns:")
print(
    con.execute(f"""
        SELECT *
        FROM read_csv(
            '{CANDIDATE_PAIRS_TSV}',
            delim='\\t',
            header=true
        )
        LIMIT 5
    """).df()
)

      rows  unique_s1
0  1732544    1732544

Columns:
  source1_entity_id                               candidate_entity_ids
0      S1-100000556  S2-773905311,S3-502189990,S3-912715604,S3-9722...
1      S1-100001013  S2-10397384,S2-104809182,S2-10581168,S2-115780...
2      S1-100001295  S2-1204764,S2-121215514,S2-162683030,S2-171851...
3      S1-100001834  S2-157626042,S2-717302325,S2-752481952,S2-9009...
4      S1-100003037  S2-109184533,S2-109581724,S2-117509078,S2-1494...


In [97]:
print(con.execute(f"""
WITH predicted AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_parquet('{TEST_S2_SCORED}')
    WHERE combined_score >= 0.87

    UNION

    SELECT source1_entity_id, candidate_entity_id
    FROM read_parquet('{TEST_S3_SCORED}')
    WHERE combined_score >= 0.87
),
candidate_file AS (
    SELECT
        source1_entity_id,
        UNNEST(
            CASE
                WHEN candidate_entity_ids = '' THEN []
                ELSE string_split(candidate_entity_ids, ',')
            END
        ) AS candidate_entity_id
    FROM read_csv(
        '{CANDIDATE_PAIRS_TSV}',
        delim='\\t',
        header=true
    )
)
SELECT COUNT(*) AS missing_predictions
FROM predicted p
LEFT JOIN candidate_file c
  ON p.source1_entity_id = c.source1_entity_id
 AND p.candidate_entity_id = c.candidate_entity_id
WHERE c.candidate_entity_id IS NULL
""").df())

   missing_predictions
0                    0


In [98]:
MATCHING_RESULTS_TSV = SUBMISSION_DIR / "matching_results.tsv"

if MATCHING_RESULTS_TSV.exists():
    MATCHING_RESULTS_TSV.unlink()

MATCH_BATCH = 50_000

s1_count = con.execute("SELECT COUNT(*) FROM test_s1").fetchone()[0]
num_match_batches = (s1_count + MATCH_BATCH - 1) // MATCH_BATCH

first_batch = True

for batch_id in range(num_match_batches):
    offset = batch_id * MATCH_BATCH

    print(f"[{batch_id + 1}/{num_match_batches}]")

    df = con.execute(f"""
        WITH batch_s1 AS (
            SELECT entity_id
            FROM test_s1
            ORDER BY entity_id
            LIMIT {MATCH_BATCH}
            OFFSET {offset}
        ),
        predictions AS (
            SELECT
                source1_entity_id,
                candidate_entity_id
            FROM read_parquet('{TEST_S2_SCORED}')
            WHERE combined_score >= 0.87

            UNION

            SELECT
                source1_entity_id,
                candidate_entity_id
            FROM read_parquet('{TEST_S3_SCORED}')
            WHERE combined_score >= 0.87
        )
        SELECT
            b.entity_id AS source1_entity_id,
            COALESCE(
                string_agg(
                    p.candidate_entity_id,
                    ',' ORDER BY p.candidate_entity_id
                ),
                ''
            ) AS matched_entity_ids
        FROM batch_s1 b
        LEFT JOIN predictions p
          ON b.entity_id = p.source1_entity_id
        GROUP BY b.entity_id
        ORDER BY b.entity_id
    """).df()

    df.to_csv(
        MATCHING_RESULTS_TSV,
        sep="\t",
        index=False,
        header=first_batch,
        mode="w" if first_batch else "a"
    )

    first_batch = False

    print(f"    rows written: {len(df):,}")

[1/35]
    rows written: 50,000
[2/35]
    rows written: 50,000
[3/35]
    rows written: 50,000
[4/35]
    rows written: 50,000
[5/35]
    rows written: 50,000
[6/35]
    rows written: 50,000
[7/35]
    rows written: 50,000
[8/35]
    rows written: 50,000
[9/35]
    rows written: 50,000
[10/35]
    rows written: 50,000
[11/35]
    rows written: 50,000
[12/35]
    rows written: 50,000
[13/35]
    rows written: 50,000
[14/35]
    rows written: 50,000
[15/35]
    rows written: 50,000
[16/35]
    rows written: 50,000
[17/35]
    rows written: 50,000
[18/35]
    rows written: 50,000
[19/35]
    rows written: 50,000
[20/35]
    rows written: 50,000
[21/35]
    rows written: 50,000
[22/35]
    rows written: 50,000
[23/35]
    rows written: 50,000
[24/35]
    rows written: 50,000
[25/35]
    rows written: 50,000
[26/35]
    rows written: 50,000
[27/35]
    rows written: 50,000
[28/35]
    rows written: 50,000
[29/35]
    rows written: 50,000
[30/35]
    rows written: 50,000
[31/35]
    rows wr

In [99]:
print(
    con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT source1_entity_id) AS unique_s1,
            SUM(
                CASE
                    WHEN matched_entity_ids = '' THEN 1
                    ELSE 0
                END
            ) AS empty_matches
        FROM read_csv(
            '{MATCHING_RESULTS_TSV}',
            delim='\\t',
            header=true
        )
    """).df()
)

print("\nSample:")
print(
    con.execute(f"""
        SELECT *
        FROM read_csv(
            '{MATCHING_RESULTS_TSV}',
            delim='\\t',
            header=true
        )
        LIMIT 5
    """).df()
)

      rows  unique_s1  empty_matches
0  1732544    1732544            0.0

Sample:
  source1_entity_id                                 matched_entity_ids
0      S1-100000556  S2-773905311,S3-502189990,S3-912715604,S3-9722...
1      S1-100001013  S2-135727199,S2-308589095,S2-534176148,S2-7851...
2      S1-100001295  S3-111745699,S3-303694164,S3-380201812,S3-8301...
3      S1-100001834                          S2-900957143,S3-420734787
4      S1-100003037  S2-207887213,S2-41163623,S2-628954077,S2-84427...


In [ ]:
print(
    con.execute(f"""
        SELECT *
        FROM read_csv(
            '{CANDIDATE_PAIRS_TSV}',
            delim='\\t',
            header=true
        )
        LIMIT 5
    """).df()
)

print(
    "Rows:",
    con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv(
            '{CANDIDATE_PAIRS_TSV}',
            delim='\\t',
            header=true
        )
    """).fetchone()[0]
)

In [102]:
import subprocess

validator = r"utils\validate_submission.py"

result = subprocess.run(
    [
        "python",
        validator,
        "--matching", str(MATCHING_RESULTS_TSV),
        "--candidate", str(CANDIDATE_PAIRS_TSV),
        "--test-dir", r"D:\business_entity_resolution\dataset\test",
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Exit code:", result.returncode)

ML Challenge 2026 â€” submission validator
  test dir: D:\business_entity_resolution\dataset\test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (620059 empty, 1112485 non-empty).
  candidate_pairs.tsv: 1732544 rows (406486 empty, 1326058 non-empty).

PASS â€” no blocking issues found. Safe to submit.

Exit code: 0
